# BBC News NLP Classifier
## A Traditional NLP Pipeline which is buit using TF-IDF, LDA, spaCy NER

This notebook builds a full NLP pipeline on the BBC News dataset.
- **Phase 1:** Sub-category discovery using TF-IDF + LDA
- **Phase 2:** Named Entity Recognition using spaCy
- **Phase 3:** April event extraction using rule-based matching

In [1]:
# Core libraries
import os
import re

# Data handling
import pandas as pd
import numpy as np

# NLP: Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# NLP: Topic Modelling
from sklearn.decomposition import LatentDirichletAllocation

# NLP: Named Entity Recognition
import spacy

# Display
from rich.console import Console
from rich.table import Table

# Load our data loader utility
from loader import load_data

##  Load the Data

In [3]:
# Load all articles and their category labels
documents, labels = load_data('data')

# Quick sanity check
print(f"Total articles loaded: {len(documents)}")
print(f"Total labels loaded: {len(labels)}")
print(f"\nCategories found: {set(labels)}")
print(f"\nSample article label: {labels[0]}")
print(f"Sample article preview:\n{documents[0][:200]}")

Total articles loaded: 2225
Total labels loaded: 2225

Categories found: {'business', 'politics', 'entertainment', 'tech', 'sport'}

Sample article label: entertainment
Sample article preview:
Musicians to tackle US red tape

Musicians' groups are to tackle US visa regulations which are blamed for hindering British acts' chances of succeeding across the Atlantic.

A singer hoping to perform


##  Data Cleaning with Regex

In [4]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def clean_text(text):
    # Lowercase everything
    text = text.lower()
    
    # Remove text inside square brackets
    text = re.sub(r'\[.*?\]', '', text)
    
    # Remove non-letter characters (numbers, punctuation, symbols)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    
    return ' '.join(tokens)

In [5]:
#Test that the cleaning function is working
sample = documents[1]
cleaned = clean_text(sample)

print("BEFORE:")
print(sample[:200])
print("\nAFTER:")
print(cleaned[:200])

BEFORE:
U2's desire to be number one

U2, who have won three prestigious Grammy Awards for their hit Vertigo, are stubbornly clinging to their status as one of the biggest bands in the world.

The most popula

AFTER:
desire number u won prestigious grammy awards hit vertigo stubbornly clinging status biggest bands world popular groups history rock things common music inspired appeal generations distinctive groundb


In [6]:
# Apply clean_text to every article in our dataset
cleaned_documents = [clean_text(doc) for doc in documents]

# Verify
print(f"Total cleaned documents: {len(cleaned_documents)}")
print(f"\nSample cleaned article:\n{cleaned_documents[0][:200]}")

Total cleaned documents: 2225

Sample cleaned article:
musicians tackle red tape musicians groups tackle visa regulations blamed hindering british acts chances succeeding atlantic singer hoping perform expect pay simply obtaining visa groups including mus


## Data Inspection & Quality Check

In [7]:
import os

# Create cleaned_data folder with subfolders per category
output_base = 'cleaned_data'

for category in set(labels):
    os.makedirs(os.path.join(output_base, category), exist_ok=True)

# Save each cleaned document into its matching category folder
for i, (doc, label) in enumerate(zip(cleaned_documents, labels)):
    file_path = os.path.join(output_base, label, f"article_{i}.txt")
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(doc)

print("Cleaned data saved successfully")
print("\nFolder structure:")
for category in sorted(set(labels)):
    count = labels.count(category)
    print(f"  cleaned_data/{category}/ — {count} articles")

Cleaned data saved successfully

Folder structure:
  cleaned_data/business/ — 510 articles
  cleaned_data/entertainment/ — 386 articles
  cleaned_data/politics/ — 417 articles
  cleaned_data/sport/ — 511 articles
  cleaned_data/tech/ — 401 articles


## Duplicate Check

In [8]:
# Check for duplicate articles in cleaned documents
total = len(cleaned_documents)
unique = len(set(cleaned_documents))
duplicates = total - unique

print(f"Total articles: {total}")
print(f"Unique articles: {unique}")
print(f"Duplicate articles found: {duplicates}")

# If duplicates exist, show a sample
if duplicates > 0:
    seen = set()
    for i, doc in enumerate(cleaned_documents):
        if doc in seen:
            print(f"\nDuplicate found at index {i} — label: {labels[i]}")
            print(f"Preview: {doc[:100]}")
            break
        seen.add(doc)
else:
    print("\nNo duplicates found. Data is clean.")

Total articles: 2225
Unique articles: 2119
Duplicate articles found: 106

Duplicate found at index 23 — label: entertainment
Preview: brits return keane number brits success helped return keanes awardwinning album hopes fears uk album


## Step 3: TF-IDF Feature Extraction

In [ ]:
\#initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=1000,
    max_df=0.95,
    min_df=2
)


#Fit and transform the cleaned documents into a TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(cleaned_documents)

#check the shape of the matrix
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")



TypeError: TfidfVectorizer.__init__() got an unexpected keyword argument 'max_feature'